# 🏗️ IFC Building Analyzer — Multi-Agentes com CrewAI

**Master Internacional em IA para Arquitetura e Construção — Zigurat Institute of Technology**

**Tema 2 — Agentic AI | Atividade Final**

---

## 🎯 Objetivo

Sistema multi-agente usando **CrewAI** integrado com **IfcOpenShell** que lê um modelo `.ifc` de edifício residencial e responde perguntas sobre áreas, materiais e quantitativos, gerando um relatório técnico `.docx` e um checklist `.xlsx`.

### Arquitetura do Sistema

```
┌─────────────────────────────────────────────────────────────┐
│                     CREW IFC ANALYZER                        │
│                                                              │
│  🔍 Agent 1: Analista BIM/IFC                                │
│     Role: Extrair e analisar dados do modelo IFC             │
│     Tools: ler_modelo_ifc, extrair_quantitativos,            │
│            extrair_espacos, extrair_materiais                │
│     ↓                                                        │
│  📝 Agent 2: Relator Técnico AEC                             │
│     Role: Sintetizar análise em relatório profissional        │
│     Tools: gerar_relatorio_docx, gerar_checklist_xlsx        │
│     ↓                                                        │
│  🧑‍💼 Agent 3: Revisor de Conformidade                        │
│     Role: Verificar conformidade e emitir parecer final       │
│     Tools: ler_modelo_ifc                                    │
│                                                              │
│  Process: SEQUENTIAL (Agent 1 → Agent 2 → Agent 3)          │
└─────────────────────────────────────────────────────────────┘
```

### Competências Demonstradas
- **C1**: Arquitetura multi-agente CrewAI (Agent → Task → Crew → Process)
- **C2**: Role, Goal, Backstory, LLM, Tools e Memory para cada agente
- **C3**: Tasks com descrição, output esperado e agente responsável
- **C4**: Orquestração com Process.sequential
- **C5**: Integração de ferramentas de leitura IFC nos agentes via BaseTool
- **C6**: Geração de entregáveis técnicos (.docx e .xlsx)


In [ ]:
# ============================================================
# CÉLULA 1 — Instalação de dependências
# ============================================================

!pip install crewai crewai-tools langchain-anthropic langchain -q
!pip install python-docx openpyxl ifcopenshell -q

# Verificação
import importlib
deps = ["crewai", "langchain_anthropic", "docx", "openpyxl", "ifcopenshell"]
for dep in deps:
    try:
        importlib.import_module(dep)
        print(f"✅ {dep}")
    except ImportError:
        print(f"❌ {dep} — execute: pip install {dep}")


In [ ]:
# ============================================================
# CÉLULA 2 — Configuração da API Key
# ============================================================
import os
from getpass import getpass

# Opção A: Colab Secrets (recomendado)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Anthropic API Key carregada via Colab Secrets")
except Exception:
    # Opção B: variável de ambiente já definida
    if "ANTHROPIC_API_KEY" not in os.environ:
        # Opção C: input manual
        os.environ["ANTHROPIC_API_KEY"] = getpass("🔑 Cole sua Anthropic API Key: ")
    print("✅ Anthropic API Key configurada")

key = os.environ.get("ANTHROPIC_API_KEY", "")
if key.startswith("sk-ant-"):
    print(f"   Chave: sk-ant-...{key[-6:]} (válida)")
else:
    print("⚠️  Formato inesperado — verifique a chave")


In [ ]:
# ============================================================
# CÉLULA 3 — Imports gerais e LLM
# ============================================================
import os
import json
import time
from pathlib import Path
from typing import Optional, Any
from datetime import datetime

# CrewAI
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

# Processamento de documentos
from docx import Document as DocxDocument
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# IFC (Open BIM)
import ifcopenshell
import ifcopenshell.util.element
print(f"✅ IfcOpenShell {ifcopenshell.version}")

# ── LLM Configuration ──
LLM_SONNET = LLM(
    model="anthropic/claude-sonnet-4-5-20250929",
    api_key=os.environ["ANTHROPIC_API_KEY"],
    temperature=0.1,
    max_tokens=4096,
)

print("\n✅ Todos os imports concluídos")
print(f"   LLM: claude-sonnet-4-5 | temp=0.1 | max_tokens=4096")
print(f"   Framework: CrewAI")


In [ ]:
# ============================================================
# CÉLULA 4 — Carregar o arquivo IFC
# ============================================================

# Diretório de trabalho
WORK_DIR = Path("./output")
WORK_DIR.mkdir(parents=True, exist_ok=True)

IFC_PATH = "casa_residencial.ifc"
assert Path(IFC_PATH).exists(), f"❌ Arquivo IFC não encontrado: {IFC_PATH}"

# Carrega o modelo
IFC_MODEL = ifcopenshell.open(IFC_PATH)
print(f"✅ Modelo IFC carregado: {IFC_PATH}")
print(f"   Schema: {IFC_MODEL.schema}")

for proj in IFC_MODEL.by_type("IfcProject"):
    print(f"   Projeto: {proj.Name}")

print("\n📊 Contagem de Entidades:")
for et in ["IfcWall","IfcDoor","IfcWindow","IfcSlab","IfcColumn","IfcBeam","IfcStair","IfcSpace","IfcRailing"]:
    n = len(IFC_MODEL.by_type(et))
    if n:
        print(f"   {et}: {n}")


---

## 🔧 Ferramentas (Tools) dos Agentes

Cada agente recebe ferramentas específicas (subclasses de `BaseTool` do CrewAI).  
As tools encapsulam operações IfcOpenShell e geração de documentos.

| Tool | Agente | Função |
|------|--------|--------|
| `LerModeloIFCTool` | Analista + Revisor | Extrai resumo hierárquico do modelo IFC |
| `ExtrairQuantitativosTool` | Analista | Extrai quantitativos de paredes, portas, janelas, estrutura |
| `ExtrairEspacosTool` | Analista | Extrai ambientes com áreas e volumes |
| `ExtrairMateriaisTool` | Analista | Extrai materiais e seus usos por tipo de elemento |
| `GerarRelatorioDocxTool` | Relator | Gera relatório .docx profissional |
| `GerarChecklistXlsxTool` | Relator | Gera checklist de conformidade .xlsx |


In [ ]:
# ============================================================
# CÉLULA 5 — Tool: Ler Modelo IFC (resumo hierárquico)
# ============================================================

class LerModeloIFCTool(BaseTool):
    name: str = "ler_modelo_ifc"
    description: str = (
        "Lê o modelo Open BIM (.ifc) e extrai resumo hierárquico: "
        "Projeto > Site > Edificação > Pavimentos > Entidades > Espaços > Propriedades. "
        "Use para ter uma visão geral do modelo antes de análises detalhadas."
    )

    def _run(self, query: str = "") -> str:
        ifc = IFC_MODEL
        resultado = ["=== MODELO IFC (Open BIM) ==="]
        resultado.append(f"Schema: {ifc.schema}")
        
        for proj in ifc.by_type("IfcProject"):
            resultado.append(f"Projeto: {proj.Name}")
        for bldg in ifc.by_type("IfcBuilding"):
            resultado.append(f"Edificação: {bldg.Name}")

        resultado.append("\n--- Contagem de Entidades ---")
        for et in ["IfcBuildingStorey","IfcWall","IfcColumn","IfcBeam","IfcSlab",
                    "IfcWindow","IfcDoor","IfcStair","IfcSpace","IfcRailing"]:
            entities = ifc.by_type(et)
            if entities:
                resultado.append(f"  {et}: {len(entities)}")

        resultado.append("\n--- Pavimentos ---")
        for storey in ifc.by_type("IfcBuildingStorey"):
            resultado.append(f"  {storey.Name}: elevação={storey.Elevation}m")

        resultado.append("\n--- Ambientes (IfcSpace) ---")
        for space in ifc.by_type("IfcSpace"):
            props = {}
            for rel in ifc.by_type("IfcRelDefinesByProperties"):
                if space in rel.RelatedObjects:
                    pset = rel.RelatingPropertyDefinition
                    if hasattr(pset, "HasProperties"):
                        for prop in pset.HasProperties:
                            if hasattr(prop, "NominalValue") and prop.NominalValue:
                                props[prop.Name] = prop.NominalValue.wrappedValue
            area = props.get("GrossFloorArea", "N/A")
            resultado.append(f"  {space.Name}: área={area} m²")

        return "\n".join(resultado)

print("✅ LerModeloIFCTool definida")
tool_ifc = LerModeloIFCTool()
print(f"   Preview: {tool_ifc._run()[:200]}...")


In [ ]:
# ============================================================
# CÉLULA 6 — Tool: Extrair Quantitativos Detalhados
# ============================================================

class ExtrairQuantitativosTool(BaseTool):
    name: str = "extrair_quantitativos"
    description: str = (
        "Extrai quantitativos detalhados do modelo IFC: paredes (dimensões, áreas, volumes), "
        "portas e janelas (dimensões, materiais), elementos estruturais (colunas, vigas, lajes). "
        "Retorna dados tabulados para análise e geração de relatório."
    )

    def _run(self, query: str = "") -> str:
        ifc = IFC_MODEL
        resultado = {"walls": [], "doors": [], "windows": [], "columns": [], "beams": [], "slabs": []}

        def get_props(element):
            info = {}
            for rel in ifc.by_type("IfcRelDefinesByProperties"):
                if element in rel.RelatedObjects:
                    pset = rel.RelatingPropertyDefinition
                    if hasattr(pset, "Quantities"):
                        for q in pset.Quantities:
                            if q.is_a("IfcQuantityLength"): info[q.Name] = round(q.LengthValue, 2)
                            elif q.is_a("IfcQuantityArea"): info[q.Name] = round(q.AreaValue, 2)
                            elif q.is_a("IfcQuantityVolume"): info[q.Name] = round(q.VolumeValue, 3)
                    elif hasattr(pset, "HasProperties"):
                        for prop in pset.HasProperties:
                            if hasattr(prop, "NominalValue") and prop.NominalValue:
                                info[prop.Name] = prop.NominalValue.wrappedValue
            mat = ifcopenshell.util.element.get_material(element)
            if mat and hasattr(mat, "Name"): info["material"] = mat.Name
            return info

        for w in ifc.by_type("IfcWall"):
            d = {"name": w.Name}; d.update(get_props(w)); resultado["walls"].append(d)
        for d in ifc.by_type("IfcDoor"):
            info = {"name": d.Name, "width": d.OverallWidth, "height": d.OverallHeight}
            if d.OverallWidth and d.OverallHeight: info["area"] = round(d.OverallWidth * d.OverallHeight, 2)
            info.update(get_props(d)); resultado["doors"].append(info)
        for w in ifc.by_type("IfcWindow"):
            info = {"name": w.Name, "width": w.OverallWidth, "height": w.OverallHeight}
            if w.OverallWidth and w.OverallHeight: info["area"] = round(w.OverallWidth * w.OverallHeight, 2)
            info.update(get_props(w)); resultado["windows"].append(info)
        for c in ifc.by_type("IfcColumn"):
            d = {"name": c.Name}; d.update(get_props(c)); resultado["columns"].append(d)
        for b in ifc.by_type("IfcBeam"):
            d = {"name": b.Name}; d.update(get_props(b)); resultado["beams"].append(d)
        for s in ifc.by_type("IfcSlab"):
            d = {"name": s.Name}; d.update(get_props(s)); resultado["slabs"].append(d)

        return json.dumps(resultado, indent=2, ensure_ascii=False)

print("✅ ExtrairQuantitativosTool definida")


In [ ]:
# ============================================================
# CÉLULA 7 — Tool: Extrair Espaços e Áreas
# ============================================================

class ExtrairEspacosTool(BaseTool):
    name: str = "extrair_espacos_areas"
    description: str = (
        "Extrai todos os ambientes (IfcSpace) do modelo com áreas brutas, áreas líquidas, "
        "volumes e pavimento associado. Calcula totais e eficiência espacial."
    )

    def _run(self, query: str = "") -> str:
        ifc = IFC_MODEL
        spaces = []
        for sp in ifc.by_type("IfcSpace"):
            info = {"name": sp.Name}
            for rel in ifc.by_type("IfcRelAggregates"):
                if sp in rel.RelatedObjects:
                    info["storey"] = rel.RelatingObject.Name
            for rel in ifc.by_type("IfcRelDefinesByProperties"):
                if sp in rel.RelatedObjects:
                    pset = rel.RelatingPropertyDefinition
                    if hasattr(pset, "Quantities"):
                        for q in pset.Quantities:
                            if q.is_a("IfcQuantityArea"): info[q.Name] = round(q.AreaValue, 2)
                            elif q.is_a("IfcQuantityVolume"): info[q.Name] = round(q.VolumeValue, 2)
                            elif q.is_a("IfcQuantityLength"): info[q.Name] = round(q.LengthValue, 2)
                    elif hasattr(pset, "HasProperties"):
                        for prop in pset.HasProperties:
                            if hasattr(prop, "NominalValue") and prop.NominalValue:
                                info[prop.Name] = prop.NominalValue.wrappedValue
            spaces.append(info)

        total_gross = sum(s.get("GrossFloorArea", 0) for s in spaces if isinstance(s.get("GrossFloorArea",0),(int,float)))
        total_net = sum(s.get("NetFloorArea", 0) for s in spaces if isinstance(s.get("NetFloorArea",0),(int,float)))
        
        return json.dumps({
            "spaces": spaces,
            "total_gross_area_m2": round(total_gross, 1),
            "total_net_area_m2": round(total_net, 1),
            "efficiency_pct": round(total_net / total_gross * 100, 1) if total_gross > 0 else 0
        }, indent=2, ensure_ascii=False)

print("✅ ExtrairEspacosTool definida")


In [ ]:
# ============================================================
# CÉLULA 8 — Tool: Extrair Materiais
# ============================================================

class ExtrairMateriaisTool(BaseTool):
    name: str = "extrair_materiais"
    description: str = (
        "Extrai resumo de todos os materiais utilizados no modelo IFC: "
        "nome do material, quantidade de elementos que o utilizam e tipos de elemento."
    )

    def _run(self, query: str = "") -> str:
        ifc = IFC_MODEL
        materials = {}
        for rel in ifc.by_type("IfcRelAssociatesMaterial"):
            mat = rel.RelatingMaterial
            mat_name = mat.Name if hasattr(mat, "Name") else "Unknown"
            if mat_name not in materials:
                materials[mat_name] = {"element_count": 0, "element_types": {}}
            for obj in rel.RelatedObjects:
                materials[mat_name]["element_count"] += 1
                etype = obj.is_a().replace("Ifc", "")
                materials[mat_name]["element_types"][etype] = materials[mat_name]["element_types"].get(etype, 0) + 1
        
        return json.dumps(materials, indent=2, ensure_ascii=False)

print("✅ ExtrairMateriaisTool definida")


In [ ]:
# ============================================================
# CÉLULA 9 — Tool: Gerar Relatório DOCX
# ============================================================

class GerarRelatorioDocxTool(BaseTool):
    name: str = "gerar_relatorio_docx"
    description: str = (
        "Gera relatório técnico profissional em formato .docx a partir da análise do modelo IFC. "
        "Input esperado: JSON com campos 'titulo', 'resumo_executivo', 'secoes' "
        "(lista de dicts com 'heading' e 'texto'), e 'recomendacoes' (lista de strings). "
        "Salva o arquivo .docx no diretório de trabalho."
    )

    def _run(self, conteudo_json: str) -> str:
        try:
            data = json.loads(conteudo_json)
        except json.JSONDecodeError:
            data = {"titulo": "Relatório de Análise IFC", 
                    "resumo_executivo": conteudo_json,
                    "secoes": [], "recomendacoes": []}

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = str(WORK_DIR / f"relatorio_analise_ifc_{ts}.docx")
        
        doc = DocxDocument()
        
        # Title page
        for _ in range(3): doc.add_paragraph("")
        t = doc.add_paragraph()
        t.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = t.add_run(data.get("titulo", "Relatório de Análise IFC"))
        run.font.size = Pt(26); run.font.color.rgb = RGBColor(0, 51, 102); run.bold = True
        
        sub = doc.add_paragraph()
        sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = sub.add_run(f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
        run.font.size = Pt(11); run.font.color.rgb = RGBColor(128,128,128)
        
        m = doc.add_paragraph()
        m.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = m.add_run("Análise automática por CrewAI + IfcOpenShell")
        run.font.size = Pt(10); run.italic = True
        
        doc.add_page_break()
        
        # Executive summary
        doc.add_heading("1. Resumo Executivo", level=1)
        doc.add_paragraph(data.get("resumo_executivo", ""))
        
        # Sections
        for i, secao in enumerate(data.get("secoes", []), 2):
            doc.add_heading(f"{i}. {secao.get('heading', '')}", level=1)
            texto = secao.get("texto", "")
            if isinstance(texto, list):
                for item in texto:
                    doc.add_paragraph(str(item))
            else:
                doc.add_paragraph(str(texto))
        
        # Recommendations
        if data.get("recomendacoes"):
            doc.add_heading("Recomendações", level=1)
            for j, rec in enumerate(data["recomendacoes"], 1):
                doc.add_paragraph(f"{j}. {rec}")
        
        # Methodology
        doc.add_page_break()
        doc.add_heading("Anexo: Metodologia", level=1)
        doc.add_paragraph(
            "Este relatório foi gerado automaticamente por um sistema multi-agente CrewAI: "
            "(1) Agente Analista BIM/IFC extraiu dados do modelo via IfcOpenShell; "
            "(2) Agente Relator Técnico sintetizou a análise em formato profissional; "
            "(3) Agente Revisor verificou conformidade e emitiu parecer final."
        )
        
        doc.save(out_path)
        size = Path(out_path).stat().st_size
        return f"✅ Relatório salvo: {out_path} ({size/1024:.1f} KB)"

print("✅ GerarRelatorioDocxTool definida")


In [ ]:
# ============================================================
# CÉLULA 10 — Tool: Gerar Checklist XLSX
# ============================================================

class GerarChecklistXlsxTool(BaseTool):
    name: str = "gerar_checklist_xlsx"
    description: str = (
        "Gera checklist de verificação em formato .xlsx. "
        "Input esperado: JSON com 'itens' (lista de dicts com "
        "'categoria', 'item', 'status', 'observacao'). "
        "Status pode ser: 'Conforme', 'Não Conforme', 'Parcial', 'N/A'. "
        "Salva o arquivo .xlsx no diretório de trabalho."
    )

    def _run(self, conteudo_json: str) -> str:
        try:
            data = json.loads(conteudo_json)
        except json.JSONDecodeError:
            data = {"itens": []}

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = str(WORK_DIR / f"checklist_ifc_{ts}.xlsx")

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Checklist IFC"
        
        # Header styling
        header_fill = PatternFill(start_color="003366", end_color="003366", fill_type="solid")
        header_font = Font(color="FFFFFF", bold=True, size=11)
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )
        
        # Status colors
        status_colors = {
            "Conforme": "C6EFCE",
            "Não Conforme": "FFC7CE",
            "Parcial": "FFEB9C",
            "N/A": "D9D9D9"
        }
        
        headers = ["#", "Categoria", "Item Verificado", "Status", "Observação"]
        for col, h in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col, value=h)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")
            cell.border = thin_border
        
        for row_i, item in enumerate(data.get("itens", []), 2):
            ws.cell(row=row_i, column=1, value=row_i-1).border = thin_border
            ws.cell(row=row_i, column=2, value=item.get("categoria", "")).border = thin_border
            ws.cell(row=row_i, column=3, value=item.get("item", "")).border = thin_border
            
            status = item.get("status", "N/A")
            status_cell = ws.cell(row=row_i, column=4, value=status)
            status_cell.border = thin_border
            status_cell.alignment = Alignment(horizontal="center")
            if status in status_colors:
                status_cell.fill = PatternFill(start_color=status_colors[status], 
                                                end_color=status_colors[status], fill_type="solid")
            
            ws.cell(row=row_i, column=5, value=item.get("observacao", "")).border = thin_border
        
        # Column widths
        ws.column_dimensions["A"].width = 5
        ws.column_dimensions["B"].width = 20
        ws.column_dimensions["C"].width = 45
        ws.column_dimensions["D"].width = 15
        ws.column_dimensions["E"].width = 50
        
        wb.save(out_path)
        size = Path(out_path).stat().st_size
        n_items = len(data.get("itens", []))
        n_conf = sum(1 for i in data.get("itens",[]) if i.get("status") == "Conforme")
        n_nc = sum(1 for i in data.get("itens",[]) if i.get("status") == "Não Conforme")
        return (f"✅ Checklist salvo: {out_path} ({size/1024:.1f} KB)\n"
                f"   Total: {n_items} itens | Conformes: {n_conf} | Não Conformes: {n_nc}")

print("✅ GerarChecklistXlsxTool definida")
print("\n📦 Todas as 6 tools criadas com sucesso!")


---

## 🤖 Definição dos 3 Agentes

Cada agente tem identidade, objetivo, backstory, LLM, tools e memória.

```
┌─────────────────────────────────────────────────────┐
│              CREW IFC ANALYZER                       │
│                                                      │
│  🔍 Analista BIM/IFC  → Extrai dados do .ifc         │
│       │                                              │
│  📝 Relator Técnico   → Gera .docx e .xlsx           │
│       │                                              │
│  🧑‍💼 Revisor           → Verifica e emite parecer     │
└─────────────────────────────────────────────────────┘
```


In [ ]:
# ============================================================
# CÉLULA 11 — Instanciar Tools
# ============================================================

tool_ler_ifc      = LerModeloIFCTool()
tool_quantitativos = ExtrairQuantitativosTool()
tool_espacos      = ExtrairEspacosTool()
tool_materiais    = ExtrairMateriaisTool()
tool_relatorio    = GerarRelatorioDocxTool()
tool_checklist    = GerarChecklistXlsxTool()

print("✅ 6 tools instanciadas")


In [ ]:
# ============================================================
# CÉLULA 12 — Agente 1: Analista BIM/IFC
# ============================================================

agente_analista = Agent(
    role="Analista BIM/IFC Sênior",
    goal=(
        "Extrair e analisar sistematicamente todos os dados do modelo IFC da edificação residencial: "
        "contagem de elementos por tipo (paredes, portas, janelas, colunas, vigas, lajes), "
        "dimensões e quantitativos (áreas, volumes, comprimentos), ambientes com áreas brutas e líquidas, "
        "e materiais utilizados com suas aplicações. "
        "Produzir um dataset estruturado e completo para subsidiar o relatório técnico."
    ),
    backstory=(
        "Você é um Arquiteto BIM Manager com certificação buildingSMART e 10 anos de experiência "
        "na implementação de metodologia BIM. Domina o padrão IFC (Industry Foundation Classes), "
        "especialmente IFC4, e sabe extrair e interpretar dados de modelos usando IfcOpenShell. "
        "Sua especialidade é transformar dados brutos de modelos BIM em informações estruturadas "
        "para tomada de decisão em projetos AEC. Você é meticuloso: nunca estima — sempre extrai "
        "os valores reais do modelo."
    ),
    llm=LLM_SONNET,
    tools=[tool_ler_ifc, tool_quantitativos, tool_espacos, tool_materiais],
    memory=True,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agente 1 criado: Analista BIM/IFC")
print(f"   Tools: {[t.name for t in agente_analista.tools]}")


In [ ]:
# ============================================================
# CÉLULA 13 — Agente 2: Relator Técnico AEC
# ============================================================

agente_relator = Agent(
    role="Relator Técnico AEC e Documentação de Projetos",
    goal=(
        "Receber os dados extraídos pelo Analista BIM e produzir dois entregáveis profissionais: "
        "(1) Relatório Técnico .docx com resumo executivo, análise de quantitativos, análise de espaços, "
        "análise de materiais e recomendações; "
        "(2) Checklist .xlsx de verificação com status de conformidade para cada aspecto do modelo. "
        "Os documentos devem ser utilizáveis por profissionais AEC sem necessidade de edição."
    ),
    backstory=(
        "Você é um Engenheiro Civil com 15 anos de experiência em documentação técnica de obras "
        "residenciais de médio e alto padrão. Especialista em elaboração de relatórios técnicos, "
        "laudos de inspeção e checklists de conformidade. Você produz documentos claros, objetivos "
        "e profissionais que são referência no setor. Sempre inclui dados numéricos concretos, "
        "tabelas comparativas e recomendações acionáveis. Você usa as tools de salvamento para "
        "gerar os arquivos — nunca apenas descreve o que faria."
    ),
    llm=LLM_SONNET,
    tools=[tool_relatorio, tool_checklist],
    memory=True,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agente 2 criado: Relator Técnico AEC")
print(f"   Tools: {[t.name for t in agente_relator.tools]}")


In [ ]:
# ============================================================
# CÉLULA 14 — Agente 3: Revisor de Conformidade
# ============================================================

agente_revisor = Agent(
    role="Revisor e Auditor de Conformidade de Modelos BIM",
    goal=(
        "Revisar os entregáveis produzidos pelos agentes anteriores e verificar: "
        "(1) Se todos os elementos do modelo IFC foram contabilizados no relatório; "
        "(2) Se as áreas e quantitativos estão coerentes; "
        "(3) Se existem questões de conformidade (portas com largura < 0.80m, espaços sem área definida); "
        "(4) Emitir parecer final com aprovação, ressalvas ou reprovação."
    ),
    backstory=(
        "Você é um Engenheiro Civil e Auditor Técnico com 18 anos de experiência em "
        "auditorias de projetos. Conhece profundamente as normas de acessibilidade e desempenho. "
        "Seu papel é garantir que o relatório produzido reflete fielmente o modelo IFC e que "
        "nenhum problema técnico passou despercebido. Você é criterioso mas justo — identifica "
        "problemas reais, não inventa. Sempre consulta o modelo IFC via tool para verificar dados."
    ),
    llm=LLM_SONNET,
    tools=[tool_ler_ifc, tool_quantitativos],
    memory=True,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agente 3 criado: Revisor de Conformidade")
print(f"   Tools: {[t.name for t in agente_revisor.tools]}")

print("\n" + "═"*60)
print("🏗️  Todos os 3 agentes criados com sucesso!")
print("═"*60)
for ag in [agente_analista, agente_relator, agente_revisor]:
    print(f"  🤖 {ag.role}")


---

## 📋 Definição das Tasks e Sequência de Execução

```
TASK 1: Analista BIM/IFC → Extrai todos os dados do modelo
    ↓ (output como contexto)
TASK 2: Relator Técnico → Gera relatório .docx e checklist .xlsx
    ↓ (output como contexto)
TASK 3: Revisor → Verifica conformidade e emite parecer final
```


In [ ]:
# ============================================================
# CÉLULA 15 — Definição das 3 Tasks
# ============================================================

# ── Task 1: Analista extrai dados do IFC ──────────────────────
task_analise = Task(
    description=(
        "Você é o Analista BIM/IFC. Realize uma análise completa do modelo IFC "
        "da Casa Residencial Modelo.\n\n"
        "Passos obrigatórios:\n"
        "1. Use a tool 'ler_modelo_ifc' para obter o resumo hierárquico do modelo\n"
        "2. Use a tool 'extrair_quantitativos' para obter todos os quantitativos "
        "(paredes, portas, janelas, colunas, vigas, lajes) com dimensões e materiais\n"
        "3. Use a tool 'extrair_espacos_areas' para obter todos os ambientes com áreas\n"
        "4. Use a tool 'extrair_materiais' para obter o resumo de materiais\n"
        "5. Compile todos os resultados em um relatório estruturado com:\n"
        "   - Resumo geral do modelo (schema, edificação, pavimentos)\n"
        "   - Tabela de quantitativos por tipo de elemento\n"
        "   - Lista de ambientes com áreas (bruta e líquida) por pavimento\n"
        "   - Mapa de materiais e seus usos\n"
        "   - Observações técnicas relevantes (ex: portas estreitas, distribuição de espaços)"
    ),
    expected_output=(
        "Relatório de Análise IFC estruturado contendo:\n"
        "- Resumo geral do modelo com contagem de entidades\n"
        "- Quantitativos detalhados de todos os elementos (15 paredes, 9 portas, etc.)\n"
        "- Lista dos 11 ambientes com áreas brutas e líquidas por pavimento\n"
        "- Mapa dos 7 materiais utilizados e seus elementos\n"
        "- Observações técnicas sobre o modelo"
    ),
    agent=agente_analista,
)

# ── Task 2: Relator gera os entregáveis ───────────────────────
task_relatorio = Task(
    description=(
        "Você é o Relator Técnico AEC. Com base na análise do Analista BIM (contexto anterior), "
        "produza DOIS entregáveis:\n\n"
        "ENTREGÁVEL 1 — RELATÓRIO .DOCX:\n"
        "Use a tool 'gerar_relatorio_docx' com JSON contendo:\n"
        "- titulo: 'Relatório de Análise do Modelo IFC — Casa Residencial Modelo'\n"
        "- resumo_executivo: visão geral do modelo (2 parágrafos com dados numéricos)\n"
        "- secoes: lista com pelo menos 4 seções:\n"
        "  a) Quantitativos de Elementos (contagem de cada tipo com totais)\n"
        "  b) Análise de Espaços e Áreas (ambientes por pavimento com áreas)\n"
        "  c) Análise de Materiais (materiais usados e onde)\n"
        "  d) Análise Estrutural (colunas, vigas, lajes com volumes de concreto)\n"
        "- recomendacoes: pelo menos 3 recomendações técnicas concretas\n\n"
        "ENTREGÁVEL 2 — CHECKLIST .XLSX:\n"
        "Use a tool 'gerar_checklist_xlsx' com JSON contendo itens de verificação:\n"
        "- Verificar se todas as paredes externas têm material definido\n"
        "- Verificar largura mínima de portas (≥ 0.80m para acessibilidade)\n"
        "- Verificar se todos os espaços têm área definida\n"
        "- Verificar se elementos estruturais têm material Concrete C30\n"
        "- Verificar se janelas externas têm transmitância definida\n"
        "- Verificar coerência de pavimentos\n"
        "- Mínimo de 15 itens no checklist\n\n"
        "IMPORTANTE: Use AMBAS as tools. Não apenas descreva — gere os arquivos."
    ),
    expected_output=(
        "Dois arquivos gerados:\n"
        "- relatorio_analise_ifc_*.docx com resumo executivo, 4+ seções e recomendações\n"
        "- checklist_ifc_*.xlsx com 15+ itens de verificação com status e observações"
    ),
    agent=agente_relator,
    context=[task_analise],
)

# ── Task 3: Revisor emite parecer final ───────────────────────
task_revisao = Task(
    description=(
        "Você é o Revisor de Conformidade. Verifique o trabalho dos agentes anteriores:\n\n"
        "1. Use a tool 'ler_modelo_ifc' para confirmar os dados do modelo original\n"
        "2. Use a tool 'extrair_quantitativos' para verificar os números apresentados\n"
        "3. Compare os dados extraídos com o que foi reportado na análise e relatório\n"
        "4. Verifique questões de conformidade:\n"
        "   - Portas com largura < 0.80m (problema de acessibilidade)\n"
        "   - Espaços sem área líquida definida\n"
        "   - Elementos estruturais sem material definido\n"
        "   - Coerência entre número de elementos reportados vs. modelo\n"
        "5. Emita parecer final com:\n"
        "   - Status: APROVADO / APROVADO COM RESSALVAS / REPROVADO\n"
        "   - Lista de conformidades confirmadas\n"
        "   - Lista de não-conformidades ou ressalvas (se houver)\n"
        "   - Recomendações para próxima fase"
    ),
    expected_output=(
        "Parecer de Revisão contendo:\n"
        "- Status de aprovação do relatório e checklist\n"
        "- Confirmação dos dados numéricos (contagem de elementos correta)\n"
        "- Lista de não-conformidades encontradas no modelo\n"
        "- Recomendações técnicas para melhoria do modelo"
    ),
    agent=agente_revisor,
    context=[task_analise, task_relatorio],
)

print("✅ 3 Tasks definidas")
print("\n📋 Resumo das Tasks:")
for i, t in enumerate([task_analise, task_relatorio, task_revisao], 1):
    print(f"  Task {i}: {t.agent.role[:50]}")


In [ ]:
# ============================================================
# CÉLULA 16 — Montagem da Crew
# ============================================================

crew_ifc = Crew(
    agents=[agente_analista, agente_relator, agente_revisor],
    tasks=[task_analise, task_relatorio, task_revisao],
    process=Process.sequential,
    verbose=True,
    memory=False,
    output_log_file=str(WORK_DIR / "crew_execution_log.txt"),
)

print("✅ Crew IFC Analyzer montada!")
print(f"\n{'═'*60}")
print("   CONFIGURAÇÃO DA CREW")
print(f"{'═'*60}")
print(f"   Agentes: {len(crew_ifc.agents)}")
print(f"   Tasks:   {len(crew_ifc.tasks)}")
print(f"   Process: {crew_ifc.process}")
print(f"   LLM:     claude-sonnet-4-5 (todos os agentes)")
print(f"   Output:  {WORK_DIR}")
print(f"{'═'*60}")
print("\n⚠️  A execução pode levar 3-8 minutos (3 tasks × LLM calls)")


In [ ]:
# ============================================================
# CÉLULA 17 — 🚀 EXECUÇÃO DA CREW
# ============================================================

print("🚀 Iniciando a Crew IFC Analyzer")
print(f"   Horário de início: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"   Modelo: {IFC_PATH}")
print()

inicio = time.time()

resultado = crew_ifc.kickoff(
    inputs={
        "modelo": IFC_PATH,
        "projeto": "Casa Residencial Modelo — Análise de Quantitativos e Conformidade",
    }
)

duracao = time.time() - inicio
print(f"\n{'═'*60}")
print(f"✅ Crew concluída em {duracao/60:.1f} minutos")
print(f"{'═'*60}")


In [ ]:
# ============================================================
# CÉLULA 18 — Resultado Final e Arquivos Gerados
# ============================================================

print("═" * 60)
print("  PARECER FINAL DO REVISOR")
print("═" * 60)
print(resultado.raw if hasattr(resultado, "raw") else str(resultado))

print("\n" + "═" * 60)
print("  ARQUIVOS GERADOS PELA CREW")
print("═" * 60)

arquivos_gerados = list(WORK_DIR.glob("relatorio_*.docx")) + \
                   list(WORK_DIR.glob("checklist_*.xlsx")) + \
                   list(WORK_DIR.glob("*.txt"))

if not arquivos_gerados:
    arquivos_gerados = list(WORK_DIR.iterdir())

for arq in sorted(arquivos_gerados, key=lambda x: x.stat().st_mtime, reverse=True):
    size = arq.stat().st_size
    mtime = datetime.fromtimestamp(arq.stat().st_mtime).strftime("%d/%m/%Y %H:%M:%S")
    icon = {"docx": "📄", "xlsx": "📊", "txt": "📝"}.get(arq.suffix.lstrip("."), "📁")
    print(f"  {icon} {arq.name}")
    print(f"      Tamanho: {size/1024:.1f} KB  |  Criado: {mtime}")


In [ ]:
# ============================================================
# CÉLULA 19 — Outputs de cada Task individual
# ============================================================

task_names = [
    (task_analise,   "Task 1 — Análise do Modelo IFC"),
    (task_relatorio, "Task 2 — Geração de Relatório e Checklist"),
    (task_revisao,   "Task 3 — Parecer de Revisão"),
]

print("═" * 60)
print("  OUTPUTS DAS TASKS")
print("═" * 60)

for task, name in task_names:
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    if hasattr(task, "output") and task.output:
        output_str = str(task.output)
        print(output_str[:1000])
        if len(output_str) > 1000:
            print(f"  ... [+{len(output_str)-1000} caracteres]")
    else:
        print("  (sem output capturado)")


---

## 📝 Reflexão Crítica (300–500 palavras)

### Decisões de Design

Este projeto implementa uma **arquitetura sequencial de três agentes** usando CrewAI para análise de modelos IFC de edificações. A escolha do CrewAI foi motivada pela sua abstração de alto nível — definir agentes com `role`, `goal` e `backstory` é mais intuitivo que construir grafos de estado manualmente, e o `Process.sequential` garante que cada agente recebe o output dos anteriores como contexto. Cada agente possui identidade, ferramentas e objetivo claramente definidos, seguindo o padrão arquitetural Agent → Task → Crew → Process apresentado na disciplina.

O design das tools segue **decomposição funcional por domínio**: em vez de uma única tool monolítica, criamos 6 tools especializadas — 4 de leitura/extração (`ler_modelo_ifc`, `extrair_quantitativos`, `extrair_espacos_areas`, `extrair_materiais`) e 2 de geração de output (`gerar_relatorio_docx`, `gerar_checklist_xlsx`). Essa separação espelha o workflow real de profissionais BIM e permite que cada agente use apenas as ferramentas relevantes ao seu papel.

### Limitações Técnicas

**1. Ausência de Raciocínio Geométrico:** O sistema extrai propriedades (nomes, quantitativos, materiais) mas não realiza análise geométrica. Não detecta conflitos espaciais (ex: viga interceptando duto), não verifica clearances, nem valida arcos de abertura de portas. O kernel geométrico do IfcOpenShell (OpenCASCADE) requer operações de malha que não estão encapsuladas nas tools atuais. Exemplo concreto: o sistema não consegue verificar se o corrimão da escada (IfcRailing) colide com a parede interna adjacente.

**2. Dependência de PropertySets Padrão:** O sistema assume arquivos IFC com property sets padrão IFC4 (`Pset_WallCommon`, `Qto_WallBaseQuantities`). Exportações reais do Revit ou ArchiCAD frequentemente usam property sets customizados (ex: `Revit_Type_Parameters`), resultando em extração incompleta sem alertar o usuário. Um arquivo IFC real de mercado provavelmente teria dados faltantes.

**3. Controle Limitado sobre Output do LLM:** Embora os dados de entrada sejam determinísticos (IfcOpenShell), o Agente Relator pode gerar texto com observações imprecisas sobre normas ou custos. O sistema mitiga isso com o Agente Revisor, mas não inclui validação automatizada contra os dados brutos.

### Extensões Possíveis

- **Detecção de Interferências:** Adicionar agente que use `ifcopenshell.geom` para tesselar elementos e detectar colisões entre sistemas (estrutural vs. MEP).
- **Verificação de Acessibilidade (NBR 9050):** Agente que verifica larguras de portas ≥ 0.80m, corredores ≥ 1.20m, e presença de rampas — o modelo atual já identifica uma porta de 0.70m como potencial não-conformidade.
- **Comparação de Versões:** Pipeline que aceita dois `.ifc` e gera relatório de diferenças por GlobalId, identificando elementos adicionados, removidos ou modificados.
- **Modo Q&A Interativo:** Substituir o pipeline sequencial por uma crew conversacional que permite perguntas de acompanhamento (ex: "Qual a área total de vidro na fachada norte?").
